# 03 – Baseline Models: ARIMA, Prophet, XGBoost

This notebook trains and evaluates three baseline forecasting models:
- **ARIMA** – classical statistical time-series model
- **Prophet** – Facebook/Meta time-series model with trend/seasonality decomposition
- **XGBoost** – gradient-boosted trees with engineered features

Predictions are saved for use in notebook 05 (ensemble).

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plotter = Plotter()

In [2]:
# ── Load processed data ─────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

target_col = 'Close'
y_test = test[target_col]

Train: 1006 | Val: 252 | Test: 249


## A. ARIMA

In [3]:
# ── ARIMA Model ──────────────────────────────────────────────────────────
from src.models.arima_model import ARIMAModel
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter

plotter = Plotter()
arima = ARIMAModel(order=(5, 1, 0))

print("--- Training ARIMA (Two-Phase) ---")
arima_val_preds, arima_test_preds = arima.train_and_refit(train, val, test, target_col='Close')

print("\n📊 [Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(val['Close'], arima_val_preds))

print("\n📊 [Phase 2] 2025 Test Metrics:")
print(calculate_metrics(test['Close'], arima_test_preds))

# 绘制 2025 年的预测对比图，保存在 reports/figures 下
plotter.plot_predictions_comparison(
    y_true=test['Close'],
    predictions={'ARIMA Forecast': arima_test_preds},
    dates=test.index,
    title="ARIMA 2025 Test Predictions",
    filename="arima_test_forecast.png"
)



--- Training ARIMA (Two-Phase) ---

📊 [Phase 1] 2024 Validation Metrics:
{'mse': 6956591.562187656, 'rmse': 2637.535130038585, 'mae': 2287.47961496122, 'mape': 11.556870357583406, 'directional_accuracy': 0.0796812749003984}

📊 [Phase 2] 2025 Test Metrics:
{'mse': 6849573.9518178245, 'rmse': 2617.169072073454, 'mae': 2169.6821826549917, 'mape': 9.309685718717489, 'directional_accuracy': 0.14516129032258066}


/Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project

## B. Prophet

In [4]:
# ── 5. Prophet Model ────────────────────────────────────────────────────────
from src.models.prophet_model import ProphetModel

prophet = ProphetModel()
print("--- Training Prophet (Two-Phase) ---")
prophet_val_preds, prophet_test_preds = prophet.train_and_refit(train, val, test)

print("\n📊 [Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(val['Close'], prophet_val_preds))

print("\n📊 [Phase 2] 2025 Test Metrics:")
print(calculate_metrics(test['Close'], prophet_test_preds))

plotter.plot_predictions_comparison(
    y_true=test['Close'],
    predictions={'Prophet Forecast': prophet_test_preds},
    dates=test.index,
    title="Prophet 2025 Test Predictions",
    filename="prophet_test_forecast.png"
)

Importing plotly failed. Interactive plots will not work.
16:35:48 - cmdstanpy - INFO - Chain [1] start processing
16:35:48 - cmdstanpy - INFO - Chain [1] done processing


--- Training Prophet (Two-Phase) ---


16:35:48 - cmdstanpy - INFO - Chain [1] start processing
16:35:48 - cmdstanpy - INFO - Chain [1] done processing



📊 [Phase 1] 2024 Validation Metrics:
{'mse': 2676886.4307547123, 'rmse': 1636.1193204515105, 'mae': 1336.4688972235706, 'mape': 6.87528784376725, 'directional_accuracy': 0.5338645418326693}

📊 [Phase 2] 2025 Test Metrics:
{'mse': 3179660.246213027, 'rmse': 1783.1601852366002, 'mae': 1320.28126033282, 'mape': 6.3676549717667825, 'directional_accuracy': 0.592741935483871}


## C. XGBoost

In [6]:
# ── 6. XGBoost Model (Strict Time-Stepped Architecture) ────────────────────
from src.models.xgboost_model import XGBoostModel
import pandas as pd

target_col = 'Close'
# 1. 抓取所有特征，但依然剔除当前行的目标变量（防同日作弊）
base_features = [col for col in train.columns if col not in [target_col, 'Adj Close']]

print("Engineering Lag-1 features to predict Tomorrow using Today's data...")

# 2. 【核心修复点：满足你的直觉！】
# 我们强制把当天的 'Close' 也加入基础特征池中！
base_features.append('Close')

# 3. 为 Train, Val, Test 创建专门的 XGBoost 数据集（将所有特征强行推迟1天）
X_train_xgb = train[base_features].shift(1)
X_val_xgb = val[base_features].shift(1)
X_test_xgb = test[base_features].shift(1)

# 由于推迟1天，第一行会变成 NaN，我们用向后填充（bfill）兜底
X_train_xgb.bfill(inplace=True)
X_val_xgb.bfill(inplace=True)
X_test_xgb.bfill(inplace=True)

# 为了防止特征重名混淆，给这些列统一打上 _lag1 的标签
xgb_feature_cols = [f"{col}_lag1" for col in base_features]
X_train_xgb.columns = xgb_feature_cols
X_val_xgb.columns = xgb_feature_cols
X_test_xgb.columns = xgb_feature_cols

# 把原始的目标变量 y 拼回去
train_xgb_df = pd.concat([X_train_xgb, train[target_col]], axis=1)
val_xgb_df = pd.concat([X_val_xgb, val[target_col]], axis=1)
test_xgb_df = pd.concat([X_test_xgb, test[target_col]], axis=1)

print(f"XGBoost is now safely learning from {len(xgb_feature_cols)} past-day features (including 'Close_lag1')!")

# 4. 开始两阶段训练
xgb = XGBoostModel()
print("--- Training XGBoost (Two-Phase Refitting) ---")
# 调用你封装好的 train_and_refit，传入我们刚刚做好的专属滞后数据集
xgb_val_preds, xgb_test_preds = xgb.train_and_refit(
    train_xgb_df, val_xgb_df, test_xgb_df,
    features=xgb_feature_cols,
    target_col=target_col
)

print("\n📊 [Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(val['Close'], xgb_val_preds))

print("\n📊 [Phase 2] 2025 Test Metrics:")
print(calculate_metrics(test['Close'], xgb_test_preds))

plotter.plot_predictions_comparison(
    y_true=test['Close'],
    predictions={'XGBoost Forecast': xgb_test_preds},
    dates=test.index,
    title="XGBoost 2025 Test Predictions",
    filename="xgboost_test_forecast.png"
)

plotter.plot_feature_importance(
    pd.Series(xgb._model.feature_importances_, index=xgb_feature_cols),
    title="XGBoost Feature Importance (Refitted)",
    filename="xgboost_feature_importance.png"
)

Engineering Lag-1 features to predict Tomorrow using Today's data...
XGBoost is now safely learning from 31 past-day features (including 'Close_lag1')!
--- Training XGBoost (Two-Phase Refitting) ---

📊 [Phase 1] 2024 Validation Metrics:
{'mse': 8067224.991659876, 'rmse': 2840.286075672638, 'mae': 2524.5490683903768, 'mape': 12.806414487350763, 'directional_accuracy': 0.4940239043824701}

📊 [Phase 2] 2025 Test Metrics:
{'mse': 14194495.256272037, 'rmse': 3767.5582618284798, 'mae': 3211.879172941767, 'mape': 13.536628127144446, 'directional_accuracy': 0.4637096774193548}


## D. Comparison

In [8]:
# ── 7. Aggregate and Save Final Test Metrics ─────────────────────────────
from src.utils.metrics import calculate_metrics
import pandas as pd
from src.config import RESULTS_DIR

# 重新计算各模型在 Phase 2 (2025 测试集) 上的最终表现
arima_metrics = calculate_metrics(test['Close'], arima_test_preds)
prophet_metrics = calculate_metrics(test['Close'], prophet_test_preds)
xgb_metrics = calculate_metrics(test['Close'], xgb_test_preds)

all_metrics = {
    'ARIMA': arima_metrics,
    'Prophet': prophet_metrics,
    'XGBoost': xgb_metrics,
}

# 转换为 DataFrame 方便展示和保存 (转置 T 是为了让模型名在行，指标在列)
metrics_df = pd.DataFrame(all_metrics).T

print("\n🏆 FINAL BASELINE METRICS (2025 Test Set - Refitted) 🏆")
print("=" * 70)
print(metrics_df.to_string())
print("=" * 70)

# 保存最终的指标表格
metrics_df.to_csv(RESULTS_DIR / 'baseline_metrics.csv')
print(f"\n✅ Metrics successfully saved to {RESULTS_DIR / 'baseline_metrics.csv'}")

# 可选：绘制所有 Baseline 模型的指标对比柱状图
plotter.plot_metrics_comparison(all_metrics, filename='baseline_metrics_comparison.png')


🏆 FINAL BASELINE METRICS (2025 Test Set - Refitted) 🏆
                  mse         rmse          mae       mape  directional_accuracy
ARIMA    6.849574e+06  2617.169072  2169.682183   9.309686              0.145161
Prophet  3.179660e+06  1783.160185  1320.281260   6.367655              0.592742
XGBoost  1.419450e+07  3767.558262  3211.879173  13.536628              0.463710

✅ Metrics successfully saved to /Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/reports/results/baseline_metrics.csv


In [10]:
# ── 8. Save Phase 1 & Phase 2 Predictions ────────────────────────────────
from src.config import RESULTS_DIR
import pandas as pd

# 1. 横向拼接各个模型的结果
val_preds_df = pd.concat([arima_val_preds, prophet_val_preds, xgb_val_preds], axis=1)
test_preds_df = pd.concat([arima_test_preds, prophet_test_preds, xgb_test_preds], axis=1)

# 2. 纵向拼接 2024(Val) 和 2025(Test) 的结果，合并为你原来熟悉的 preds_df
preds_df = pd.concat([val_preds_df, test_preds_df], axis=0)
# 重命名列，保持规范
preds_df.columns = ['ARIMA', 'Prophet', 'XGBoost']

# 3. 完美还原你原来的保存逻辑和打印语句
save_path = RESULTS_DIR / 'baseline_predictions.csv'
preds_df.to_csv(save_path)

print(f"Predictions successfully saved to {save_path}")
print(f"Total rows saved: {len(preds_df)} (Val: {len(val_preds_df)} + Test: {len(test_preds_df)})")

Predictions successfully saved to /Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/reports/results/baseline_predictions.csv
Total rows saved: 501 (Val: 252 + Test: 249)


## Summary

See `reports/results/baseline_metrics.csv` for a full metrics table.

Continue to **04_model_lstm.ipynb** for the deep learning model.